# 03_observation_window_policy_260513

관측창과 대응기간 계약을 고정하는 감사 notebook입니다.

이 단계에서는 modeling, prediction, SHAP, Optuna, feature engineering, row exclusion, duplicate removal을 수행하지 않습니다.

In [1]:
from pathlib import Path
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED
import re

import pandas as pd
import numpy as np

PARK_ROOT = Path(r"C:\\Code\\ott-churn-prediction\\park.ingyeom").resolve()
SOURCE_PATH = PARK_ROOT / "data" / "(광일)Membership_v2_with_derived_features.csv"
PREV_01_DIR = PARK_ROOT / "reports" / "audits" / "01_data_contract_260513"
PREV_02_DIR = PARK_ROOT / "reports" / "audits" / "02_target_score_orientation_260513"
NOTEBOOK_PATH = PARK_ROOT / "notebook" / "03_observation_window_policy_260513" / "03_observation_window_policy_260513.ipynb"
BASE_OUTPUT_DIR = PARK_ROOT / "reports" / "audits" / "03_observation_window_policy_260513"
NOTE_PATH = PARK_ROOT / "note.md"
ZIP_DIR = PARK_ROOT / "zip"
ZIP_PATH = ZIP_DIR / "03_observation_window_policy_260513_review_package.zip"

def is_inside(child: Path, parent: Path) -> bool:
    try:
        child.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False

BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if any(BASE_OUTPUT_DIR.iterdir()):
    OUTPUT_DIR = BASE_OUTPUT_DIR / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
else:
    OUTPUT_DIR = BASE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

for p in [SOURCE_PATH, PREV_01_DIR, PREV_02_DIR, NOTEBOOK_PATH, OUTPUT_DIR, NOTE_PATH, ZIP_PATH]:
    assert is_inside(p, PARK_ROOT), f"Path outside park.ingyeom: {p}"

source_exists = SOURCE_PATH.exists()
previous_01_folder_exists = PREV_01_DIR.exists()
previous_02_folder_exists = PREV_02_DIR.exists()
source_stat_before = SOURCE_PATH.stat() if source_exists else None
note_stat_before = NOTE_PATH.stat() if NOTE_PATH.exists() else None
written_files = []
warnings = []

print("SOURCE_PATH:", SOURCE_PATH)
print("PREV_01_DIR:", PREV_01_DIR)
print("PREV_02_DIR:", PREV_02_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("ZIP_PATH:", ZIP_PATH)

SOURCE_PATH: C:\Code\ott-churn-prediction\park.ingyeom\data\(광일)Membership_v2_with_derived_features.csv
PREV_01_DIR: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513
PREV_02_DIR: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513
OUTPUT_DIR: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513
ZIP_PATH: C:\Code\ott-churn-prediction\park.ingyeom\zip\03_observation_window_policy_260513_review_package.zip


In [2]:
def save_csv(frame: pd.DataFrame, filename: str):
    path = OUTPUT_DIR / filename
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output: {path}")
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    written_files.append(path)
    print("saved:", path)
    return path

def read_prev_csv(path: Path, label: str):
    if path.exists():
        try:
            return pd.read_csv(path), "FOUND"
        except Exception as exc:
            warnings.append(f"previous output read failed: {label}: {exc}")
            return None, "WARNING_READ_FAILED"
    warnings.append(f"previous output missing: {label}")
    return None, "WARNING_MISSING"

prev_sources = {
    "01_data_contract_summary.csv": PREV_01_DIR / "01_data_contract_summary.csv",
    "01_date_parse_audit.csv": PREV_01_DIR / "01_date_parse_audit.csv",
    "01_duration_anomaly_audit.csv": PREV_01_DIR / "01_duration_anomaly_audit.csv",
    "01_expected_vs_actual_checks.csv": PREV_01_DIR / "01_expected_vs_actual_checks.csv",
    "02_target_contract.csv": PREV_02_DIR / "02_target_contract.csv",
    "02_score_orientation_policy.csv": PREV_02_DIR / "02_score_orientation_policy.csv",
    "02_final_checks.csv": PREV_02_DIR / "02_final_checks.csv",
}
prev_data = {}
prev_status_rows = []
for label, path in prev_sources.items():
    frame, status = read_prev_csv(path, label)
    prev_data[label] = frame
    prev_status_rows.append({"item": f"previous_output::{label}", "actual_value": status, "status": status, "note": str(path)})

df = pd.read_csv(SOURCE_PATH)
row_count = int(len(df))
column_count = int(df.shape[1])
total_missing_count = int(df.isna().sum().sum())
unique_user_key_count = int(df["USER_KEY"].nunique(dropna=True)) if "USER_KEY" in df.columns else np.nan
duplicated_user_key_extra_rows = int(row_count - unique_user_key_count) if "USER_KEY" in df.columns else np.nan
duplicated_full_row_count = int(df.duplicated().sum())

target_distribution = df["is_repurchase"].value_counts(dropna=False).rename_axis("is_repurchase").reset_index(name="count") if "is_repurchase" in df.columns else pd.DataFrame()
if not target_distribution.empty:
    target_distribution["rate"] = target_distribution["count"] / row_count
promotion_distribution = df["is_promotion"].value_counts(dropna=False).rename_axis("is_promotion").reset_index(name="count") if "is_promotion" in df.columns else pd.DataFrame()
if not promotion_distribution.empty:
    promotion_distribution["rate"] = promotion_distribution["count"] / row_count
if {"is_promotion", "is_repurchase"}.issubset(df.columns):
    promotion_target_2x2 = df.groupby(["is_promotion", "is_repurchase"], dropna=False).size().reset_index(name="count")
    promotion_target_2x2["row_total_by_is_promotion"] = promotion_target_2x2.groupby("is_promotion", dropna=False)["count"].transform("sum")
    promotion_target_2x2["row_percentage_within_is_promotion"] = promotion_target_2x2["count"] / promotion_target_2x2["row_total_by_is_promotion"]
else:
    promotion_target_2x2 = pd.DataFrame()

parsed_reg = pd.to_datetime(df["reg_date"], errors="coerce") if "reg_date" in df.columns else pd.Series([pd.NaT] * row_count)
parsed_end = pd.to_datetime(df["end_date"], errors="coerce") if "end_date" in df.columns else pd.Series([pd.NaT] * row_count)
reg_parse_success = int(parsed_reg.notna().sum())
reg_parse_failure = int(df["reg_date"].notna().sum() - reg_parse_success) if "reg_date" in df.columns else np.nan
end_parse_success = int(parsed_end.notna().sum())
end_parse_failure = int(df["end_date"].notna().sum() - end_parse_success) if "end_date" in df.columns else np.nan
duration_days = (parsed_end - parsed_reg).dt.days
duration_lt_21 = duration_days < 21
duration_eq_0 = duration_days == 0
duration_1_20 = (duration_days >= 1) & (duration_days <= 20)
duration_21_30 = (duration_days >= 21) & (duration_days <= 30)
duration_gte_21 = duration_days >= 21

def mask_count(mask):
    return int(mask.fillna(False).sum())

actual_metrics = {
    "row_count": row_count,
    "column_count": column_count,
    "total_missing_count": total_missing_count,
    "unique_USER_KEY_count": unique_user_key_count,
    "duplicated_USER_KEY_extra_rows": duplicated_user_key_extra_rows,
    "duplicated_full_row_count": duplicated_full_row_count,
    "reg_date_parse_success_count": reg_parse_success,
    "reg_date_parse_failure_count": reg_parse_failure,
    "end_date_parse_success_count": end_parse_success,
    "end_date_parse_failure_count": end_parse_failure,
    "duration_lt_21_count": mask_count(duration_lt_21),
    "duration_eq_0_count": mask_count(duration_eq_0),
    "duration_21_30_count": mask_count(duration_21_30),
    "duration_gte_21_count": mask_count(duration_gte_21),
}

prev_metric_map = {}
prev_summary = prev_data.get("01_data_contract_summary.csv")
if prev_summary is not None and {"metric", "value"}.issubset(prev_summary.columns):
    for _, row in prev_summary.iterrows():
        prev_metric_map[str(row["metric"])] = row["value"]
prev_expected = prev_data.get("01_expected_vs_actual_checks.csv")
if prev_expected is not None and {"check_name", "actual_value"}.issubset(prev_expected.columns):
    for _, row in prev_expected.iterrows():
        prev_metric_map[str(row["check_name"])] = row["actual_value"]

input_rows = [
    {"item": "source_file_exists", "actual_value": bool(SOURCE_PATH.exists()), "previous_value": "", "match_previous": "", "status": "PASS" if SOURCE_PATH.exists() else "FAIL", "note": str(SOURCE_PATH)},
    {"item": "previous_01_folder_exists", "actual_value": bool(PREV_01_DIR.exists()), "previous_value": "", "match_previous": "", "status": "PASS" if PREV_01_DIR.exists() else "WARNING", "note": str(PREV_01_DIR)},
    {"item": "previous_02_folder_exists", "actual_value": bool(PREV_02_DIR.exists()), "previous_value": "", "match_previous": "", "status": "PASS" if PREV_02_DIR.exists() else "WARNING", "note": str(PREV_02_DIR)},
]
for key, actual in actual_metrics.items():
    prev_key = key
    if key == "duplicated_USER_KEY_extra_rows":
        prev_key = "duplicated_USER_KEY_row_count"
    prev = prev_metric_map.get(prev_key, "not found in previous outputs")
    try:
        match = float(actual) == float(prev)
    except Exception:
        match = "not_compared"
    input_rows.append({"item": key, "actual_value": actual, "previous_value": prev, "match_previous": match, "status": "RECORDED", "note": "recomputed from source CSV"})
input_consistency = pd.DataFrame(input_rows + prev_status_rows)

display(input_consistency)
save_csv(input_consistency, "03_input_consistency_check.csv")

,item,actual_value,previous_value,match_previous,status,note
0,source_file_exists,True,,,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...
1,previous_01_folder_exists,True,,,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
2,previous_02_folder_exists,True,,,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
3,row_count,23343,23343,True,RECORDED,recomputed from source CSV
4,column_count,91,91,True,RECORDED,recomputed from source CSV
5,total_missing_count,0,0,True,RECORDED,recomputed from source CSV
6,unique_USER_KEY_count,23134,23134,True,RECORDED,recomputed from source CSV
7,duplicated_USER_KEY_extra_rows,209,209,True,RECORDED,recomputed from source CSV
8,duplicated_full_row_count,48,48,True,RECORDED,recomputed from source CSV
9,reg_date_parse_success_count,23343,not found in previous outputs,not_compared,RECORDED,recomputed from source CSV


saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\03_input_consistency_check.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/03_observation_window_policy_260513/03_input_consistency_check.csv')

In [3]:
observation_window_policy = pd.DataFrame([
    {"policy_item": "anchor_date", "definition": "reg_date", "allowed_use": "subscription start anchor", "caution": "must parse successfully"},
    {"policy_item": "day_0", "definition": "day 0 is reg_date", "allowed_use": "relative day calculation", "caution": "absolute dates vary by row"},
    {"policy_item": "week_1", "definition": "day 0 to day 6", "allowed_use": "timing-policy candidate feature period", "caution": "not yet feature engineering"},
    {"policy_item": "week_2", "definition": "day 7 to day 13", "allowed_use": "timing-policy candidate feature period", "caution": "not yet feature engineering"},
    {"policy_item": "week_3", "definition": "day 14 to day 20", "allowed_use": "timing-policy candidate feature period", "caution": "not yet feature engineering"},
    {"policy_item": "scoring_point", "definition": "day 21", "allowed_use": "future score assignment point", "caution": "no score created in this step"},
    {"policy_item": "response_period", "definition": "day 21 to before subscription end", "allowed_use": "business action period", "caution": "response-period behavior must not become model feature"},
    {"policy_item": "target_observation", "definition": "next-month repurchase proxy through is_repurchase", "allowed_use": "target only", "caution": "not a causal outcome"},
    {"policy_item": "forbidden_feature_period", "definition": "4th week / response-period behavior", "allowed_use": "forbidden for modeling features", "caution": "timing leakage and operational logic risk"},
])

timing_allowed_for_modeling_policy = pd.DataFrame([
    {"timing_family": "week1_or_w1", "policy": "yes", "meaning": "candidate if constructed only from day 0 to day 6", "caution": "timing audit still needed"},
    {"timing_family": "week2_or_w2", "policy": "yes", "meaning": "candidate if constructed only from day 7 to day 13", "caution": "timing audit still needed"},
    {"timing_family": "week3_or_w3", "policy": "yes", "meaning": "candidate if constructed only from day 14 to day 20", "caution": "timing audit still needed"},
    {"timing_family": "week4_or_w4", "policy": "no", "meaning": "4th-week/response-period behavior", "caution": "forbidden for modeling"},
    {"timing_family": "total_or_all_period", "policy": "review", "meaning": "total/all-period naming does not prove day 0-20 only", "caution": "must verify construction window"},
    {"timing_family": "retention_or_diff", "policy": "review", "meaning": "ratio/diff needs source window confirmation", "caution": "may mix periods"},
    {"timing_family": "recency", "policy": "review", "meaning": "recency needs timestamp/window proof", "caution": "may reference late behavior"},
    {"timing_family": "content_without_explicit_week", "policy": "review", "meaning": "content proxy without explicit observation window", "caution": "must document construction window"},
    {"timing_family": "membership_or_static", "policy": "review", "meaning": "static/context candidate", "caution": "business/timing leakage review still required for some columns"},
    {"timing_family": "id_target_split_date", "policy": "no", "meaning": "id, target, split, or raw date fields", "caution": "not model features"},
    {"timing_family": "unknown_review_required", "policy": "review", "meaning": "cannot classify from name alone", "caution": "manual review required"},
])

save_csv(observation_window_policy, "03_observation_window_policy.csv")
save_csv(timing_allowed_for_modeling_policy, "03_timing_allowed_for_modeling_policy.csv")

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\03_observation_window_policy.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\03_timing_allowed_for_modeling_policy.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/03_observation_window_policy_260513/03_timing_allowed_for_modeling_policy.csv')

In [4]:
def classify_column(col: str):
    c = col.lower()
    if c in {"user_key", "is_repurchase", "is_promotion", "reg_date", "end_date"} or "date" in c or c.endswith("_id") or c in {"id", "target"}:
        return "id_target_split_date", "no", "identifier, target, split variable, or raw date field"
    if re.search(r"(^|[_-])(w4|week4|week_4|4w)([_-]|$)", c) or "4th" in c:
        return "week4_or_w4", "no", "explicit 4th-week or response-period behavior is forbidden"
    if re.search(r"(^|[_-])(w1|week1|week_1|1w)([_-]|$)", c):
        return "week1_or_w1", "yes", "explicit week 1 naming; timing-policy candidate only"
    if re.search(r"(^|[_-])(w2|week2|week_2|2w)([_-]|$)", c):
        return "week2_or_w2", "yes", "explicit week 2 naming; timing-policy candidate only"
    if re.search(r"(^|[_-])(w3|week3|week_3|3w)([_-]|$)", c):
        return "week3_or_w3", "yes", "explicit week 3 naming; timing-policy candidate only"
    if any(tok in c for tok in ["total", "overall", "all_period", "allperiod", "all_"]):
        return "total_or_all_period", "review", "total/all-period name does not prove day 0-20 only"
    if any(tok in c for tok in ["retention", "diff", "delta", "ratio_change", "change"]):
        return "retention_or_diff", "review", "retention/diff feature needs construction window proof"
    if any(tok in c for tok in ["recency", "recent", "last", "latest"]):
        return "recency", "review", "recency feature needs timing proof"
    if any(tok in c for tok in ["genre", "content", "movie", "watch"]):
        return "content_without_explicit_week", "review", "content/behavior column has no explicit week in name"
    if any(tok in c for tok in ["age", "gender", "price", "billing", "payment", "screen", "product", "promotion", "churn_prevented", "verified"]):
        return "membership_or_static", "review", "static/context column; later business/timing review required"
    return "unknown_review_required", "review", "cannot classify from name alone"

feature_inventory_rows = []
for col in df.columns:
    family, allowed, reason = classify_column(col)
    feature_inventory_rows.append({
        "column_name": col,
        "dtype": str(df[col].dtype),
        "timing_family": family,
        "likely_allowed_by_observation_policy": allowed,
        "reason": reason,
    })
week_feature_inventory = pd.DataFrame(feature_inventory_rows)
display(week_feature_inventory)
save_csv(week_feature_inventory, "03_week_feature_inventory.csv")

,column_name,dtype,timing_family,likely_allowed_by_observation_policy,reason
0,USER_KEY,object,id_target_split_date,no,"identifier, target, split variable, or raw dat..."
1,product_code,object,membership_or_static,review,static/context column; later business/timing r...
2,price,float64,membership_or_static,review,static/context column; later business/timing r...
3,billing_method,int64,membership_or_static,review,static/context column; later business/timing r...
4,max_screen,float64,membership_or_static,review,static/context column; later business/timing r...
...,...,...,...,...,...
86,romance_ratio,float64,unknown_review_required,review,cannot classify from name alone
87,horror_ratio,float64,unknown_review_required,review,cannot classify from name alone
88,documentary_ratio,float64,unknown_review_required,review,cannot classify from name alone
89,historical_war_ratio,float64,unknown_review_required,review,cannot classify from name alone


saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\03_week_feature_inventory.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/03_observation_window_policy_260513/03_week_feature_inventory.csv')

In [5]:
duration_distribution = duration_days.value_counts(dropna=False).rename_axis("duration_days").reset_index(name="count")
duration_distribution["rate"] = duration_distribution["count"] / row_count
duration_distribution = duration_distribution.sort_values("duration_days", na_position="last")

def anomaly_summary_rows(prefix, data, group_cols=None):
    masks = {
        "duration_lt_21": duration_lt_21,
        "duration_eq_0": duration_eq_0,
        "duration_1_20": duration_1_20,
        "duration_21_30": duration_21_30,
        "duration_gte_21": duration_gte_21,
    }
    rows = []
    if not group_cols:
        for name, mask in masks.items():
            cnt = mask_count(mask)
            rows.append({"grouping": "overall", "group_value": "all", "metric": name, "row_count": row_count, "count": cnt, "rate": cnt / row_count})
        return pd.DataFrame(rows)
    for group_values, idx in data.groupby(group_cols, dropna=False).groups.items():
        if not isinstance(group_values, tuple):
            group_values = (group_values,)
        group_label = "|".join([f"{c}={v}" for c, v in zip(group_cols, group_values)])
        group_index = list(idx)
        denom = len(group_index)
        for name, mask in masks.items():
            cnt = int(mask.loc[group_index].fillna(False).sum())
            rows.append({"grouping": "|".join(group_cols), "group_value": group_label, "metric": name, "row_count": denom, "count": cnt, "rate": cnt / denom if denom else np.nan})
    return pd.DataFrame(rows)

duration_group_tables = [anomaly_summary_rows("overall", df)]
if "is_promotion" in df.columns:
    duration_group_tables.append(anomaly_summary_rows("promotion", df, ["is_promotion"]))
if "is_repurchase" in df.columns:
    duration_group_tables.append(anomaly_summary_rows("target", df, ["is_repurchase"]))
if {"is_promotion", "is_repurchase"}.issubset(df.columns):
    duration_group_tables.append(anomaly_summary_rows("promotion_target", df, ["is_promotion", "is_repurchase"]))
duration_anomaly_by_group = pd.concat(duration_group_tables, ignore_index=True)

duration_policy_options = pd.DataFrame([
    {"option_name": "Option 1: Keep all rows for now", "description": "No row exclusion in step 03", "benefit": "Preserves source population for policy review", "risk": "duration < 21 may mix insufficient observation time with low behavior", "recommended_for_current_goal": "Y", "recommended_for_future_step": "review", "reason": "current goal is policy documentation only"},
    {"option_name": "Option 2: Exclude duration < 21 from main modeling cohort later", "description": "Potential future main cohort excludes observation-window-incomplete rows", "benefit": "Aligns main modeling with 1-3 week observation design", "risk": "may remove meaningful short-duration cases without business review", "recommended_for_current_goal": "N", "recommended_for_future_step": "candidate", "reason": "final include/exclude policy belongs to later cohort/final feature step"},
    {"option_name": "Option 3: Keep duration < 21 as separate anomaly segment later", "description": "Flag short-duration rows for separate analysis", "benefit": "Avoids losing anomaly cases while protecting main interpretation", "risk": "requires clear segment wording and enough sample size", "recommended_for_current_goal": "N", "recommended_for_future_step": "candidate", "reason": "step 03 only flags policy options"},
    {"option_name": "Option 4: Sensitivity analysis with and without duration < 21 later", "description": "Compare downstream results under both policies", "benefit": "Shows robustness to cohort policy", "risk": "extra work and potential narrative complexity", "recommended_for_current_goal": "N", "recommended_for_future_step": "recommended", "reason": "best future validation of duration policy impact"},
])

save_csv(duration_distribution, "03_duration_distribution.csv")
save_csv(duration_anomaly_by_group, "03_duration_anomaly_by_group.csv")
save_csv(duration_policy_options, "03_duration_policy_options.csv")

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\03_duration_distribution.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\03_duration_anomaly_by_group.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\03_duration_policy_options.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/03_observation_window_policy_260513/03_duration_policy_options.csv')

In [6]:
safe_unsafe_wording = pd.DataFrame([
    {"unsafe_expression": "3주차까지 봤으므로 초조기 예측이다.", "safer_alternative": "1~3주차 행동을 관측하고 day 21 이후 대응하는 갱신 직전 이탈 방어 모델이다.", "reason": "scoring point is day 21, not ultra-early"},
    {"unsafe_expression": "duration < 21은 고객센터 즉시해지다.", "safer_alternative": "duration < 21은 단기 종료 후보이며, 고객센터 즉시해지 여부는 CSV만으로 확정할 수 없다.", "reason": "CSV has dates but no cancellation reason"},
    {"unsafe_expression": "4주차 행동도 모델에 넣으면 성능이 좋아진다.", "safer_alternative": "4주차는 대응기간에 해당하므로 feature로 사용하면 운영 논리와 timing leakage 위험이 생긴다.", "reason": "response-period behavior is forbidden"},
    {"unsafe_expression": "duration < 21 고객은 모두 제거한다.", "safer_alternative": "duration < 21 행은 관측창 불완전 후보로 flag하고, main cohort 제외 여부는 후속 단계에서 결정한다.", "reason": "no final cohort exclusion in this goal"},
])

open_risks = pd.DataFrame([
    {"risk": "duration < 21 rows remain included for now", "carry_forward_to": "cohort policy", "current_policy": "flag only, no exclusion"},
    {"risk": "duration < 21 may confound low behavior with insufficient observation time", "carry_forward_to": "cohort policy and modeling design", "current_policy": "document as observation-window-incomplete candidates"},
    {"risk": "duration = 0 rows need business interpretation but cannot be explained from CSV alone", "carry_forward_to": "business data review", "current_policy": "do not claim immediate customer-center cancellation"},
    {"risk": "4th-week behavior must remain forbidden as feature", "carry_forward_to": "feature selection and leakage audit", "current_policy": "explicit w4/week4 columns marked no"},
    {"risk": "total/all-period columns need timing review before modeling", "carry_forward_to": "leakage/timing audit", "current_policy": "marked review"},
    {"risk": "recency needs timing review before modeling", "carry_forward_to": "leakage/timing audit", "current_policy": "marked review"},
    {"risk": "content columns without explicit observation window need timing review", "carry_forward_to": "feature construction review", "current_policy": "marked review"},
    {"risk": "end_date/duration timing still needs leakage/timing audit", "carry_forward_to": "leakage/timing audit", "current_policy": "not audited in step 03"},
    {"risk": "full duplicate rows still need policy review", "carry_forward_to": "deduplication/cohort policy", "current_policy": "no duplicate rows removed"},
    {"risk": "groupwise models later must not use is_promotion as a feature", "carry_forward_to": "promotion split modeling", "current_policy": "carry forward from step 02"},
])

save_csv(safe_unsafe_wording, "03_safe_unsafe_wording.csv")
save_csv(open_risks, "03_open_risks_for_next_steps.csv")

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\03_safe_unsafe_wording.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\03_open_risks_for_next_steps.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/03_observation_window_policy_260513/03_open_risks_for_next_steps.csv')

In [7]:
readme_text = f"""# 03_observation_window_policy_260513

This is step 03 only.

- No modeling was performed.
- No SHAP was performed.
- No Optuna was performed.
- No feature engineering was performed.
- No rows were excluded.
- No duplicated rows were removed.
- The policy is 1~3 week observation, day 21 scoring, day 21 onward response period.
- 4th-week behavior is forbidden for modeling.
- duration < 21 rows are flagged but not removed.
- Next recommended step is `04_promotion_split_260513`.

## Source

`{SOURCE_PATH}`

## Output Folder

`{OUTPUT_DIR}`
"""
readme_path = OUTPUT_DIR / "README.md"
if readme_path.exists():
    raise FileExistsError(f"Refusing to overwrite existing output: {readme_path}")
readme_path.write_text(readme_text, encoding="utf-8")
written_files.append(readme_path)
print("saved:", readme_path)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
warning_text = "none" if not warnings else "; ".join(warnings)
note_section = f"""

## {timestamp} - 03_observation_window_policy_260513

- Purpose: 1~3주차 관측창, day 21 scoring point, day 21 이후 대응기간, 4주차 금지 정책, duration anomaly 후보 정책을 문서화했다.
- Files created: notebook `notebook/03_observation_window_policy_260513/03_observation_window_policy_260513.ipynb`, output folder `{OUTPUT_DIR.relative_to(PARK_ROOT)}`, review zip `zip/03_observation_window_policy_260513_review_package.zip`.
- Key decisions: reg_date를 day 0 anchor로 두고 week1=day0~6, week2=day7~13, week3=day14~20, scoring point=day21, response period=day21~subscription end 전으로 정의했다. 4th-week behavior는 modeling feature로 금지한다.
- Checks: source CSV exists={source_exists}; previous 01 folder exists={previous_01_folder_exists}; previous 02 folder exists={previous_02_folder_exists}; row_count={row_count}; column_count={column_count}; duration_lt_21={mask_count(duration_lt_21)}; duration_eq_0={mask_count(duration_eq_0)}; duration_21_30={mask_count(duration_21_30)}; duplicated_full_row_count={duplicated_full_row_count}.
- Interpretation limits: modeling, prediction, SHAP, Optuna, feature engineering, leakage/timing audit, row exclusion, duplicate removal은 수행하지 않았다. 행을 unique user로 부르지 않는다.
- Risks to carry forward: duration < 21 rows, duration=0 interpretation, total/all-period timing, recency timing, content window proof, end_date/duration leakage, full duplicate row policy, groupwise model에서 is_promotion 제외 필요.
- Warnings: {warning_text}.
- Next step recommendation: 04_promotion_split_260513.
"""
with NOTE_PATH.open("a", encoding="utf-8") as f:
    f.write(note_section)
written_files.append(NOTE_PATH)
print("updated:", NOTE_PATH)

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\README.md
updated: C:\Code\ott-churn-prediction\park.ingyeom\note.md


In [8]:
def create_review_zip(zip_path: Path, include_final_checks: bool):
    if zip_path.exists():
        zip_path.unlink()
    csv_files = sorted(OUTPUT_DIR.glob("*.csv"))
    if not include_final_checks:
        csv_files = [p for p in csv_files if p.name != "03_final_checks.csv"]
    with ZipFile(zip_path, "w") as zf:
        zf.write(NOTEBOOK_PATH, NOTEBOOK_PATH.relative_to(PARK_ROOT))
        for p in csv_files:
            zf.write(p, p.relative_to(PARK_ROOT))
        zf.write(readme_path, readme_path.relative_to(PARK_ROOT))
        zf.write(NOTE_PATH, NOTE_PATH.relative_to(PARK_ROOT))
    return zip_path

create_review_zip(ZIP_PATH, include_final_checks=False)
written_files.append(ZIP_PATH)
print("created initial zip:", ZIP_PATH)

created initial zip: C:\Code\ott-churn-prediction\park.ingyeom\zip\03_observation_window_policy_260513_review_package.zip


In [9]:
source_stat_after = SOURCE_PATH.stat() if SOURCE_PATH.exists() else None
note_updated = NOTE_PATH.exists() and (note_stat_before is None or NOTE_PATH.stat().st_size > note_stat_before.st_size)

w4_rows = week_feature_inventory[week_feature_inventory["timing_family"] == "week4_or_w4"]
total_rows = week_feature_inventory[week_feature_inventory["timing_family"] == "total_or_all_period"]
recency_rows = week_feature_inventory[week_feature_inventory["timing_family"] == "recency"]

checks = []
def add_check(check, passed, evidence):
    checks.append({"check": check, "status": "PASS" if passed else "FAIL", "evidence": evidence})

add_check("source_file_exists", SOURCE_PATH.exists(), str(SOURCE_PATH))
add_check("source_file_inside_park_ingyeom", is_inside(SOURCE_PATH, PARK_ROOT), str(SOURCE_PATH))
add_check("previous_01_folder_exists", PREV_01_DIR.exists(), str(PREV_01_DIR))
add_check("previous_02_folder_exists", PREV_02_DIR.exists(), str(PREV_02_DIR))
add_check("notebook_inside_park_ingyeom", is_inside(NOTEBOOK_PATH, PARK_ROOT), str(NOTEBOOK_PATH))
add_check("output_folder_inside_park_ingyeom", is_inside(OUTPUT_DIR, PARK_ROOT), str(OUTPUT_DIR))
add_check("no_files_written_outside_park_ingyeom", all(is_inside(p, PARK_ROOT) for p in written_files), "all tracked writes are inside park.ingyeom")
add_check("no_py_script_created", not any(p.suffix.lower() == ".py" for p in written_files), "no tracked .py files")
add_check("no_existing_notebook_modified", True, "new step 03 notebook only")
add_check("no_source_csv_modified", source_stat_before and source_stat_after and source_stat_before.st_size == source_stat_after.st_size and source_stat_before.st_mtime == source_stat_after.st_mtime, "source size and mtime unchanged")
add_check("no_modeling_performed", True, "no estimator fit/train or model object created")
add_check("no_predictions_created", True, "no prediction column or score output created")
add_check("no_shap_performed", True, "no shap import or computation")
add_check("no_optuna_performed", True, "no optuna import or tuning")
add_check("no_feature_engineering_performed", True, "no new feature columns or model-ready dataset created")
add_check("no_rows_excluded", len(df) == row_count, "all source rows retained for audit computations")
add_check("no_duplicate_rows_removed", duplicated_full_row_count == int(df.duplicated().sum()), "duplicates only counted, not removed")
add_check("observation_window_policy_created", (OUTPUT_DIR / "03_observation_window_policy.csv").exists(), "03_observation_window_policy.csv")
add_check("week_feature_inventory_created", (OUTPUT_DIR / "03_week_feature_inventory.csv").exists(), "03_week_feature_inventory.csv")
add_check("explicit_w4_columns_marked_forbidden_if_present", w4_rows.empty or (w4_rows["likely_allowed_by_observation_policy"] == "no").all(), f"w4_columns={len(w4_rows)}")
add_check("total_all_period_columns_marked_review_if_present", total_rows.empty or (total_rows["likely_allowed_by_observation_policy"] == "review").all(), f"total_all_columns={len(total_rows)}")
add_check("recency_marked_review_if_present", recency_rows.empty or (recency_rows["likely_allowed_by_observation_policy"] == "review").all(), f"recency_columns={len(recency_rows)}")
add_check("duration_distribution_created", (OUTPUT_DIR / "03_duration_distribution.csv").exists(), "03_duration_distribution.csv")
add_check("duration_anomaly_by_group_created", (OUTPUT_DIR / "03_duration_anomaly_by_group.csv").exists(), "03_duration_anomaly_by_group.csv")
add_check("duration_policy_options_created", (OUTPUT_DIR / "03_duration_policy_options.csv").exists(), "03_duration_policy_options.csv")
add_check("safe_unsafe_wording_created", (OUTPUT_DIR / "03_safe_unsafe_wording.csv").exists(), "03_safe_unsafe_wording.csv")
add_check("open_risks_created", (OUTPUT_DIR / "03_open_risks_for_next_steps.csv").exists(), "03_open_risks_for_next_steps.csv")
add_check("readme_created", readme_path.exists(), "README.md")
add_check("note_md_updated", note_updated, str(NOTE_PATH))
add_check("review_zip_created", ZIP_PATH.exists(), str(ZIP_PATH))

required_outputs = [
    "03_input_consistency_check.csv",
    "03_observation_window_policy.csv",
    "03_week_feature_inventory.csv",
    "03_duration_distribution.csv",
    "03_duration_anomaly_by_group.csv",
    "03_duration_policy_options.csv",
    "03_timing_allowed_for_modeling_policy.csv",
    "03_safe_unsafe_wording.csv",
    "03_open_risks_for_next_steps.csv",
    "README.md",
]
for fname in required_outputs:
    add_check(f"required_output_exists::{fname}", (OUTPUT_DIR / fname).exists(), fname)

final_checks = pd.DataFrame(checks)
save_csv(final_checks, "03_final_checks.csv")
create_review_zip(ZIP_PATH, include_final_checks=True)

print("all_final_checks_passed:", bool((final_checks["status"] == "PASS").all()))
print("warnings:", warnings)
display(final_checks)

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\03_observation_window_policy_260513\03_final_checks.csv
all_final_checks_passed: True
warnings: []


,check,status,evidence
0,source_file_exists,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...
1,source_file_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...
2,previous_01_folder_exists,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
3,previous_02_folder_exists,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
4,notebook_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\note...
5,output_folder_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
6,no_files_written_outside_park_ingyeom,PASS,all tracked writes are inside park.ingyeom
7,no_py_script_created,PASS,no tracked .py files
8,no_existing_notebook_modified,PASS,new step 03 notebook only
9,no_source_csv_modified,PASS,source size and mtime unchanged


## Final Summary

### Checked items

- Source CSV and previous 01/02 audit folders.
- Minimum consistency metrics and duration anomaly counts.
- Observation window policy: day 0 anchor, week 1, week 2, week 3, scoring point, response period, target observation, forbidden 4th-week period.
- Actual CSV columns classified into timing families from their names.
- Duration anomaly distribution and grouping by promotion, target, and promotion-target combination.
- Duration policy options without applying exclusions.
- Safe and unsafe wording.
- Open risks for next steps.

### Unchecked items

- No modeling was performed.
- No prediction score was created.
- No SHAP was performed.
- No Optuna was performed.
- No feature engineering was performed.
- No leakage/timing audit was performed.
- No rows were excluded.
- No duplicated rows were removed.

### Interpretation limits

- The analysis unit remains row-level / subscription-event-level.
- Week-related columns are classified from actual names only; construction logic still requires later timing review.
- Duration < 21 rows are observation-window-incomplete candidates only, not excluded rows.
- CSV alone cannot prove why duration = 0 or duration < 21 occurred.

### Next recommended step

`04_promotion_split_260513`